# Train and quantize YOLOv8n-seg for STM32N6

Run the repository's COCO TFLite model on your PC, then fine-tune YOLOv8n-seg on your own images and export a model for the NUCLEO-N657X0-Q. [Notebook 02](02_Compile_and_Deploy_NUCLEO_N6.ipynb) takes the exported model and its class metadata through deployment.

## Python environment

- For the **COCO reference test**, install `ai-edge-litert numpy pillow` in the Jupyter environment. The cell can also use the TensorFlow interpreter if installed.
- For **training and export**, create an environment with Python 3.10 or 3.11. The [ST export tutorial](https://github.com/STMicroelectronics/stm32ai-modelzoo-services/blob/main/instance_segmentation/docs/tuto/How_to_deploy_yolov8_instance_segmentation.md) validated Ultralytics 8.3.88. Start with the versions below. The exporter checks additional conversion dependencies. Save `python -m pip freeze` once the export works.

```bash
python -m pip install "ultralytics==8.3.88" "tensorflow==2.15.0" "onnx==1.15.0" "onnxruntime==1.18.1" "keras==2.15.0" "tf-keras==2.15.0" ipykernel
python -m ipykernel install --user --name stm32n6-yolov8 --display-name "Python 3 (STM32N6 YOLOv8)"
```

Select the new kernel in Jupyter. On Windows, WSL2 is an option if the export stack causes problems. Copy the exported TFLite and class JSON to the machine that runs ST Edge AI Core.

In [ ]:
from pathlib import Path
import json
import numpy as np
from PIL import Image
from IPython.display import display

HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "Model").is_dir() else HERE.parent
# REPO = Path(r"C:\projects\STM32N6-Edge_Computer_Vision")
BASELINE = REPO / "Model" / "yolov8n_256_quant_pc_ii_seg_coco-st.tflite"
assert BASELINE.is_file(), f"Set REPO to the repository root. Missing {BASELINE}"

WORK = REPO.parent / "stm32n6_model_work"
DATASET_ROOT = REPO.parent / "insect_dataset"
CLASS_NAMES = ["insect"]  # Match the class IDs in the label files.
IMAGE_SIZE = 256
IMAGE = None  # Optional: Path(r"C:\path\capture.jpg")

## 1. Run the COCO reference model

The bundled TFLite uses signed 8-bit input with a scale of about 1/255 and a zero point of −128. ST Edge AI Core generates an unsigned 8-bit input with zero point 0 for the same model, so the firmware can feed it raw RGB bytes. The conversion is `real = scale × (quantized - zero_point)`.

This test resizes an RGB image directly to 256 × 256, matching the aspect-ratio behavior of the firmware inference pipe. Interpolation and camera ISP can still change individual pixels. To compare inference tensors on the PC and board, use the same input RGB bytes.

In [ ]:
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    from tensorflow.lite import Interpreter

def open_tflite(path):
    interpreter = Interpreter(model_path=str(path), num_threads=2)
    interpreter.allocate_tensors()
    return interpreter

def tensor_info(detail):
    scale, zero_point = detail["quantization"]
    return {
        "name": detail["name"],
        "shape": detail["shape"].tolist(),
        "dtype": np.dtype(detail["dtype"]).name,
        "scale": float(scale),
        "zero_point": int(zero_point),
    }

def run_tflite(interpreter, rgb):
    inputs = interpreter.get_input_details()
    assert len(inputs) == 1, "Expected a single RGB input."
    info = inputs[0]
    scale, zero_point = info["quantization"]
    assert scale > 0 and info["dtype"] in (np.int8, np.uint8)
    limits = np.iinfo(info["dtype"])
    real = rgb[None].astype(np.float32) / 255
    assert real.shape == tuple(info["shape"])
    data = np.clip(np.rint(real / scale + zero_point), limits.min, limits.max)
    interpreter.set_tensor(info["index"], data.astype(info["dtype"]))
    interpreter.invoke()
    return [interpreter.get_tensor(d["index"]) for d in interpreter.get_output_details()]

baseline = open_tflite(BASELINE)
for detail in [*baseline.get_input_details(), *baseline.get_output_details()]:
    print(tensor_info(detail))

if IMAGE is None:
    rgb = np.full((IMAGE_SIZE, IMAGE_SIZE, 3), 220, dtype=np.uint8)
else:
    reference = Image.open(IMAGE).convert("RGB")
    rgb = np.asarray(reference.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR))

raw_outputs = run_tflite(baseline, rgb)
print("COCO outputs:", [(output.shape, output.dtype.name) for output in raw_outputs])

## 2. Prepare images and polygons

Create `insect_dataset/images/{train,val,test}` and `insect_dataset/labels/{train,val,test}` beside the repository. For each `001.jpg`, add `001.txt` in the corresponding labels folder. Each object has one row with the class ID and the polygon coordinates normalized to the image width and height.

```text
0 0.40 0.44 0.54 0.43 0.55 0.57 0.41 0.55
```

Use an empty TXT file for a background image. Group captures by event, date, or trap before splitting. This keeps nearly identical burst frames from appearing in both train and test. Include empty backgrounds, reflections, motion, and difficult examples. Check whether insects are still visible after resizing to 256 × 256.

[Ultralytics segmentation dataset format](https://docs.ultralytics.com/datasets/segment/)

In [ ]:
import yaml

assert CLASS_NAMES and len(CLASS_NAMES) == len(set(CLASS_NAMES))
assert len(CLASS_NAMES) <= 255, "The demo postprocessor uses 8-bit class IDs."
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
first_test_image = None

for split in ("train", "val", "test"):
    images_root = DATASET_ROOT / "images" / split
    labels_root = DATASET_ROOT / "labels" / split
    images = sorted(p for p in images_root.rglob("*") if p.suffix.lower() in image_extensions)
    assert images, f"No images in {images_root}"
    instances = 0
    backgrounds = 0
    for image_path in images:
        label = labels_root / image_path.relative_to(images_root).with_suffix(".txt")
        assert label.is_file(), f"Missing {label}. Use an empty file for a background image."
        annotation = label.read_text(encoding="utf-8")
        backgrounds += not annotation.strip()
        for row_number, row in enumerate(annotation.splitlines(), 1):
            tokens = row.split()
            if not tokens:
                continue
            location = f"{label} line {row_number}"
            assert len(tokens) >= 7 and len(tokens) % 2 == 1, f"Invalid polygon: {location}"
            assert 0 <= int(tokens[0]) < len(CLASS_NAMES), f"Invalid class ID: {location}"
            assert all(0 <= float(x) <= 1 for x in tokens[1:]), f"Invalid coordinates: {location}"
            instances += 1
    assert instances, f"The {split} split has no labeled instances."
    print(f"{split}: {len(images)} images, {backgrounds} backgrounds, {instances} instances")
    if split == "test":
        first_test_image = images[0]

WORK.mkdir(exist_ok=True)
DATA_YAML = WORK / "insects.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATASET_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": dict(enumerate(CLASS_NAMES)),
}, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(DATA_YAML.read_text(encoding="utf-8"))

## 3. Train the PyTorch model

For the first iteration, keep YOLOv8n-seg and 256 × 256 input. This reduces the changes needed in the firmware. Track box and mask mAP as well as false positives on background images. Use the validation split to tune training and keep the test split for evaluating the final models.

In [ ]:
from ultralytics import YOLO

EPOCHS = 50
DEVICE = None  # Use 0 for the first CUDA GPU or "cpu" to force CPU.
trainer = YOLO("yolov8n-seg.pt")
trainer.train(
    data=str(DATA_YAML), imgsz=IMAGE_SIZE, epochs=EPOCHS,
    project=str(WORK / "runs"), name="insect_yolov8n_seg",
    device=DEVICE, workers=0, seed=42,
)
BEST_PT = Path(trainer.trainer.best).resolve()
assert BEST_PT.is_file(), BEST_PT
print(BEST_PT)

## 4. Export a fully quantized TFLite model

In the dedicated Python environment, follow ST's per-channel export workflow. Open `ultralytics/engine/exporter.py` and change the `onnx2tf.convert` argument from `quant_type="per-tensor"` to `quant_type="per-channel"`. After editing, restart the kernel and rerun the configuration, reference, and dataset cells to restore the Python variables. If you already trained the model, skip retraining and set `BEST_PT` in the export cell to the checkpoint for that run.

The exporter uses the validation split of `insects.yaml` to calibrate with your own camera images. Keep the `*_full_integer_quant.tflite` file. The next cell checks its tensor types and shapes. Changes to the output order, layout, or meaning may require corresponding changes to the C postprocessor.

In [ ]:
import inspect
from ultralytics.engine import exporter

assert 'quant_type="per-channel"' in inspect.getsource(exporter), (
    f"Set per-channel quantization in {inspect.getfile(exporter)} and restart the kernel."
)

# After restarting the kernel, set BEST_PT to the checkpoint from your training run.
# BEST_PT = WORK / "runs" / "insect_yolov8n_seg" / "weights" / "best.pt"
fp_model = YOLO(str(BEST_PT))
assert list(fp_model.names.values()) == CLASS_NAMES
exported = Path(fp_model.export(
    format="tflite", imgsz=IMAGE_SIZE, batch=1, int8=True, nms=False,
    data=str(DATA_YAML), split="val", device="cpu",
)).resolve()
QUANTIZED = exported.parent / f"{BEST_PT.stem}_full_integer_quant.tflite"
assert QUANTIZED.is_file(), f"Look for the full_integer_quant file beside {exported}"

In [ ]:
candidate = open_tflite(QUANTIZED)
input_specs = [tensor_info(d) for d in candidate.get_input_details()]
output_specs = [tensor_info(d) for d in candidate.get_output_details()]
print(json.dumps({"input": input_specs, "outputs": output_specs}, indent=2))
assert len(input_specs) == 1 and input_specs[0]["shape"] == [1, 256, 256, 3]
assert input_specs[0]["dtype"] == "int8"
assert all(spec["dtype"] == "int8" and spec["scale"] > 0 for spec in output_specs)
assert len(output_specs) == 2
expected_shapes = {(1, 4 + len(CLASS_NAMES) + 32, 1344), (1, 64, 64, 32)}
assert {tuple(spec["shape"]) for spec in output_specs} == expected_shapes, (
    "Tensor shapes differ from the demo. Check the export and C integration."
)

manifest = {
    "model": str(QUANTIZED),
    "class_names": CLASS_NAMES,
    "imgsz": IMAGE_SIZE,
    "input": input_specs,
    "outputs": output_specs,
}
MANIFEST = WORK / "insect_model.json"
MANIFEST.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(QUANTIZED)
print(MANIFEST)

## 5. Evaluate both models on the test split

Compare box and mask mAP on the same split. Then inspect both models on the same resized image. Ultralytics evaluation uses its own preprocessing. For the final check, use tunnel images and the board's real preprocessing.

In [ ]:
models = {"PyTorch FP32": YOLO(str(BEST_PT)), "TFLite INT8": YOLO(str(QUANTIZED), task="segment")}
for name, model in models.items():
    metrics = model.val(data=str(DATA_YAML), split="test", imgsz=IMAGE_SIZE, batch=1, device="cpu")
    print(name, {"box mAP50-95": metrics.box.map, "mask mAP50-95": metrics.seg.map})

sample = Image.open(first_test_image if IMAGE is None else IMAGE).convert("RGB")
sample = sample.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
for name, model in models.items():
    result = model.predict(sample, imgsz=IMAGE_SIZE, conf=0.5, iou=0.4, max_det=10, device="cpu")[0]
    print(name)
    display(Image.fromarray(result.plot()[..., ::-1]))

Pass the `.tflite` and `insect_model.json` to [Notebook 02](02_Compile_and_Deploy_NUCLEO_N6.ipynb). If they move to another machine, set the new TFLite path in that notebook. Preserve the order of `class_names` from training.